In [ ]:
import os, sys
sys.path.append(os.path.abspath(".."))  # adjust so `src` is importable

import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

from src import config
from src.data_pipeline import (
    load_merged_dataframe, get_column_groups, split_partitions,
    normalize_targets, make_dataset,
)
from src.model import build_model, compile_model
from src.training import train_chunked, train_in_sessions

print("TF version:", tf.__version__)

In [ ]:
df = load_merged_dataframe()
attr_cols, bbox_cols, landmark_cols = get_column_groups(df)
train_df, val_df, test_df = split_partitions(df)

train_df = normalize_targets(train_df, bbox_cols, landmark_cols)
val_df   = normalize_targets(val_df, bbox_cols, landmark_cols)
test_df  = normalize_targets(test_df, bbox_cols, landmark_cols)

print(f"train={len(train_df)}  val={len(val_df)}  test={len(test_df)}")
print(f"attrs={len(attr_cols)}  landmarks={len(landmark_cols)}  bbox={len(bbox_cols)}")

In [ ]:
train_ds = make_dataset(train_df, attr_cols, bbox_cols, landmark_cols, shuffle=True)
val_ds   = make_dataset(val_df,   attr_cols, bbox_cols, landmark_cols)

In [ ]:
model, backbone = build_model(
    img_size=config.IMG_SIZE,
    num_attrs=len(attr_cols),
    num_landmarks=len(landmark_cols),
    num_bbox=len(bbox_cols),
)
model = compile_model(model)
model.summary()

In [ ]:
model = train_in_sessions(
    model, train_ds, val_ds,
    callbacks=[],       # no cross-session callbacks — see explanation above
    stage_name="normal",
)

In [ ]:
import json
with open(os.path.join(config.CHECKPOINT_DIR, "state.json")) as f:
    print(json.load(f))